# Sprint 7 — Attributes, Registration, Validation & Diagnostics

Spec: [`sprint-7-tasks.md`](../../litemapper/docs/requirements/sprint-7-tasks.md) — 13 tasks (S7-T00..T12).

Sprint 7 wires the fluent surface, the assembly scanner, and the diagnostics pipeline into a shippable `SculptorBuilder` → `Sculptor` runtime. It's the sprint that makes the library *usable* end-to-end.

| Scope                                              | Task    |
| -------------------------------------------------- | ------- |
| `[MappedBy<T>]` / `[Unmapped]` / `[LinkedFrom]`    | S7-T00  |
| `AssemblyScanner`                                  | S7-T02  |
| `SculptorOptions` + `SculptorBuilder` accumulator  | S7-T03..T04 |
| Inline `options.Bind<S, D>(…)`                     | S7-T05  |
| `Forge()` freeze + runtime `Sculptor`              | S7-T06..T07 |
| `Validate()` + `StrictMode`                        | S7-T09  |
| `Inspect<S, D>()` per-link trace                   | S7-T10  |
| `MappingAtlas` + DOT export                        | S7-T11  |


## Setup


In [ ]:
#r "../src/SmartMapp.Net/bin/Release/net10.0/SmartMapp.Net.dll"
using SmartMapp.Net;
using SmartMapp.Net.Abstractions;
using SmartMapp.Net.Attributes;
using SmartMapp.Net.Diagnostics;
Console.WriteLine("Ready.");


## 1. `[MappedBy<T>]` + `[Unmapped]` + `[LinkedFrom]`

Attribute-driven configuration lets DTOs declare their mapping intent *on the type itself* — no fluent configuration needed at the registration site.

- **`[MappedBy<TOrigin>]`** — declares the origin type on the DTO.
- **`[Unmapped]`** — blocks a target member from conventional flow.
- **`[LinkedFrom("…")]`** — explicit origin member path, supports dotted navigation.


In [ ]:
public sealed class Customer    { public string FirstName { get; init; } = ""; public string LastName { get; init; } = ""; }
public sealed class S7_Order    { public int Id { get; init; } public Customer Customer { get; init; } = new(); public string InternalTag { get; init; } = ""; }

[MappedBy<S7_Order>]
public sealed class S7_OrderDto
{
    public int    Id { get; set; }

    [LinkedFrom("Customer.FirstName")]
    public string BuyerFirstName { get; set; } = "";

    [Unmapped]
    public string InternalTag { get; set; } = "";
}

// .ScanAssembliesContaining<T>() picks up [MappedBy<T>] types from the same assembly.
var sculptor = new SculptorBuilder()
    .ScanAssembliesContaining<S7_OrderDto>()
    .Forge();

var dto = sculptor.Map<S7_Order, S7_OrderDto>(new S7_Order
{
    Id = 1,
    Customer = new Customer { FirstName = "Grace", LastName = "Hopper" },
    InternalTag = "do-not-leak",
});

Console.WriteLine($"Id             = {dto.Id}");
Console.WriteLine($"BuyerFirstName = {dto.BuyerFirstName}    (via [LinkedFrom(\"Customer.FirstName\")])");
Console.WriteLine($"InternalTag    = \"{dto.InternalTag}\"   (blocked by [Unmapped])");


## 2. `Inspect<S, D>()` — per-link diagnostic trace

Every `PropertyLink` records which convention produced it, the origin path, and any explicit override. `Inspect<S, D>()` flattens that into a human-readable per-link table — the first stop for debugging *"why did this member get that value?"*.


In [ ]:
var inspection = sculptor.Inspect<S7_Order, S7_OrderDto>();

Console.WriteLine($"Blueprint: {inspection.TypePair.OriginType.Name} → {inspection.TypePair.TargetType.Name}");
Console.WriteLine($"Links    : {inspection.Links.Count}   Skipped: {inspection.SkippedMembers.Count}");
Console.WriteLine();
Console.WriteLine($"  {"Target",-16} {"Convention",-24} {"Origin path",-28}");
Console.WriteLine("  " + new string('-', 72));
foreach (var line in inspection.Links)
    Console.WriteLine($"  {line.TargetMember.Name,-16} {line.LinkedBy.ConventionName,-24} {line.LinkedBy.OriginMemberPath,-28}");

if (inspection.SkippedMembers.Count > 0)
{
    Console.WriteLine();
    Console.WriteLine("Skipped members:");
    foreach (var s in inspection.SkippedMembers) Console.WriteLine($"  • {s}");
}


## 3. `MappingAtlas` + `ToDotFormat()` — graph of every registered pair

The atlas is a directed graph (nodes = types, edges = blueprints) — useful for onboarding new developers (*"what types are in scope?"*) or rendering documentation. `ToDotFormat()` emits a Graphviz DOT string you can paste into Graphviz / an online renderer.


In [ ]:
var atlas = sculptor.GetMappingAtlas();

Console.WriteLine($"Nodes: {atlas.Nodes.Count}, Edges: {atlas.Edges.Count}");
foreach (var n in atlas.Nodes) Console.WriteLine($"  • {n}");
foreach (var e in atlas.Edges) Console.WriteLine($"  → {e.Pair.OriginType.Name} → {e.Pair.TargetType.Name}  ({e.LinkCount} link(s))");

Console.WriteLine();
Console.WriteLine("--- Graphviz DOT (paste into https://dreampuf.github.io/GraphvizOnline) ---");
Console.WriteLine(atlas.ToDotFormat());


## 4. `ValidateConfiguration()` — catch misconfigurations before shipping

Errors surface as `BlueprintValidationError` entries. Warnings (e.g. unlinked target members) surface separately. You can either call `Validate()` to throw loudly or inspect the `ValidationResult` programmatically — the DI package wraps this as an `IHostedService` (Sprint 8) for fail-fast startup.


In [ ]:
// This DTO deliberately has a target member (MysteryField) with no matching origin.
public sealed class WonkyDto
{
    public int    Id           { get; set; }
    public string MysteryField { get; set; } = "";
}

var s1 = new SculptorBuilder().Configure(o => o.Bind<S7_Order, WonkyDto>(_ => { })).Forge();

var result = ((ISculptorConfiguration)s1).ValidateConfiguration();
Console.WriteLine($"IsValid : {result.IsValid}");
Console.WriteLine($"Errors  : {result.Errors.Count}");
Console.WriteLine($"Warnings: {result.Warnings.Count}");

foreach (var w in result.Warnings.Take(5))
    Console.WriteLine($"  ⚠  {w}");


## 5. `StrictMode()` — promote warnings to errors

Opt-in: flip the switch per pair (`rule.StrictMode()`) and unlinked-target warnings become validation errors. Combined with the Sprint 8 `SculptorStartupValidator`, this gives fail-fast behaviour in Development environments.


In [ ]:
var s2 = new SculptorBuilder()
    .Configure(o => o.Bind<S7_Order, WonkyDto>(rule => rule.StrictMode()))
    .Forge();

var result = ((ISculptorConfiguration)s2).ValidateConfiguration();
Console.WriteLine($"IsValid : {result.IsValid}");
Console.WriteLine($"Errors  : {result.Errors.Count}   (StrictMode promoted warnings to errors)");
foreach (var e in result.Errors.Take(5))
    Console.WriteLine($"  ✗ {e}");


## Next

- **`sprint-08-di-projection-compose.ipynb`** — `AddSculptor`, `IMapper<>` resolution, `SelectAs<T>` EF-Core projection, multi-origin `Compose<T>`, ambient `MapTo<T>`.
